# VLM-DENTAL — YOLO Grounding Tool Cross-Validation Workspace

This notebook trains and evaluates the **YOLOv8 Tooth Grounding Tool (`locate_tooth`)** across a combined multi-institutional dataset (**DENTEX + Tufts**) using **5-Fold Cross-Validation**.

### Key Architecture Features:
- **Multi-Dataset Grounding Pool**: Combines DENTEX (~1,389 images, 21.8k tooth boxes) and Tufts (1,000 images, ~25k tooth boxes) into a unified grounding corpus of **~2,389 images (~46,800 tooth bboxes)**.
- **Parallel Multi-Colab Support (`TARGET_FOLD`)**: Train all 5 folds sequentially in one session (`TARGET_FOLD = 'all'`), or run individual folds in separate Colab tabs simultaneously (`TARGET_FOLD = 0`, `1`, `2`, `3`, `4`) in parallel (~30 mins total!).
- **Benchmark Preservation**: Evaluates on the **official 50-image held-out DENTEX test set** (`test/`) for apples-to-apples comparability with challenge leaderboards.
- **Interactive Visualizer**: Displays OPG scans with color-coded bounding boxes and two-digit FDI tooth labels.
- **Hugging Face Hub Sync**: Automatically packages training curves, confusion matrices, metrics, and weights to `Reza-Nadimi/vlm-dental-models/yolo_cv`.

## 1. Mount Google Drive & Setup Workspace
Mounts Google Drive to access `.env` credentials, clones/pulls the latest `VLM-DENTAL` repository, and sets the models output directory.

In [ ]:
import os

# ============================================================
#  TOGGLE: Set IS_COLAB = True for Google Colab, False for PC
# ============================================================
IS_COLAB = True

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    work_dir = '/content/VLM-DENTAL'
    if not os.path.exists(work_dir):
        os.chdir('/content')
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
    os.chdir(work_dir)
    os.system('git pull')
else:
    # Local PC: ensure we are inside repo
    if not os.path.exists('VLM-DENTAL') and not os.path.exists('.git'):
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
        os.chdir('VLM-DENTAL')
    elif os.path.exists('VLM-DENTAL'):
        os.chdir('VLM-DENTAL')
    os.system('git pull')
    work_dir = os.getcwd()

os.environ['YOLO_MODELS_ROOT'] = os.path.join(work_dir, 'data', 'models')
print(f'Active working directory: {os.getcwd()}')
print(f'YOLO models directory:   {os.environ["YOLO_MODELS_ROOT"]}')
print(f'Mode: {"Google Colab" if IS_COLAB else "Local PC"}')

## 2. Fast-Boot Dependency Installation
Verifies existing packages to avoid redundant pip installs, cleans up audio conflicts, and installs `ultralytics` with repository dependencies.

In [ ]:
import sys
try:
    import ultralytics
    import dotenv
    import yaml
    print('✅ Dependencies already installed. Skipping pip install. (Fast boot!)')
except ImportError:
    print('⚠️ Installing dependencies...')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[api]'])

## 3. Configure Credentials (.env as Single Source of Truth)
Loads credentials in two layers:
1. **`.env` File (Primary)**: Searches Google Drive (`/content/drive/MyDrive/VLM-DENTAL/.env`) or repository root.
2. **Colab Secrets Tab (Fallback)**: If no `.env` is found, reads `HF_TOKEN` from `google.colab.userdata`.

In [ ]:
import os, shutil
from dotenv import load_dotenv

candidate_env_paths = [
    os.path.join(os.getcwd(), '.env'),
    '/content/drive/MyDrive/VLM-DENTAL/.env',
    '/content/drive/MyDrive/vlmdental/.env',
    '/content/drive/MyDrive/.env',
    '/content/VLM-DENTAL/.env',
    '/content/.env',
]

found_env = next((p for p in candidate_env_paths if os.path.exists(p) and os.path.getsize(p) > 0), None)
if found_env:
    local_env = os.path.join(os.getcwd(), '.env')
    if os.path.abspath(found_env) != os.path.abspath(local_env):
        shutil.copy(found_env, local_env)
        print(f'Copied .env from {found_env} to {local_env}')
    load_dotenv(local_env, override=True)
    print(f'✅ Loaded credentials from single source of truth: {found_env}')
else:
    print('ℹ️ No .env file found. Checking Colab Secrets tab...')
    try:
        from google.colab import userdata
        for key in ['HF_TOKEN', 'DENTEX_IMAGES_REPO', 'TUFTS_IMAGES_REPO', 'HF_ARTIFACT_REPO']:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f'  Injected {key} from Colab Secrets')
    except Exception as e:
        print(f'  Colab Secrets lookup notice: {e}')

print(f'HF_ARTIFACT_REPO: {os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")}')

## 4. Interactive Configuration & Hyperparameters
Configure dataset sources, fold targets, training epochs, and YOLO model architecture.

> **💡 Parallel Training Tip:** To train all 5 folds simultaneously across separate Colab instances, set `TARGET_FOLD = 0` in Tab 1, `TARGET_FOLD = 1` in Tab 2, etc. Each tab auto-syncs its fold to Hugging Face upon completion!

In [ ]:
# ============================================================
#  INTERACTIVE CONFIGURATION
# ============================================================
DATASETS = "dentex,tufts"    # Multi-dataset: "dentex,tufts" (default) or "dentex"
FOLDS = 5                    # Number of cross-validation folds
TARGET_FOLD = "all"          # "all" (runs 0..4 sequentially) or specific fold index: 0, 1, 2, 3, 4

EPOCHS = 50                  # Training epochs per fold
BATCH = 16                   # Batch size (16 for T4 / A100)
IMGSZ = 640                  # Input image resolution
MODEL = "yolov8m.pt"         # Architecture: yolov8m.pt (recommended), yolov8s.pt, or yolov8n.pt
DEVICE = "0"                 # GPU Device ('0' for CUDA GPU, 'cpu' for CPU)
PATIENCE = 15                # Early stopping patience
RESUME = True                # Resume from checkpoint if interrupted

print('Configuration summary:')
print(f'  Datasets:     {DATASETS}')
print(f'  Target Fold:  {TARGET_FOLD} (Total folds: {FOLDS})')
print(f'  Model:        {MODEL}')
print(f'  Epochs/Batch: {EPOCHS} epochs, batch {BATCH}')

## 5. Prepare Multi-Dataset Cross-Validation Splits
Converts DENTEX and Tufts annotations into YOLO bounding box format (`(quadrant-1)*8 + (position-1)` -> 32 tooth classes) and constructs the 5 balanced CV fold directories.

In [ ]:
!python scripts/prepare_yolo_dataset.py --mode cv --folds {FOLDS} --datasets {DATASETS}

## 6. Train YOLO Grounding Tool (Cross-Validation)
Runs YOLOv8 training on the selected fold(s). Auto-syncs weights and performance metrics (`best.pt`, `results.csv`, `PR_curve.png`) to Hugging Face.

In [ ]:
resume_flag = '--resume' if RESUME else ''
patience_flag = f'--patience {PATIENCE}' if PATIENCE else ''

!python scripts/train_grounding_tool.py \
    --cross-validate \
    --target-fold {TARGET_FOLD} \
    --folds {FOLDS} \
    --epochs {EPOCHS} \
    --batch {BATCH} \
    --imgsz {IMGSZ} \
    --model {MODEL} \
    --device {DEVICE} \
    --datasets {DATASETS} \
    {resume_flag} \
    {patience_flag}

## 7. Held-Out Benchmark & Model Aggregation
Evaluates all 5 trained fold checkpoints against the **official 50-image DENTEX held-out test set** and compares `best.pt` vs `last.pt` performance.

In [ ]:
!python scripts/train_grounding_tool.py \
    --eval-benchmark \
    --datasets {DATASETS} \
    --batch {BATCH} \
    --imgsz {IMGSZ} \
    --device {DEVICE}

## 8. Interactive Grounding Visualizer
Visually inspects predicted bounding boxes and FDI tooth labels against ground truth on sample panoramic radiographs.

In [ ]:
import os, glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# Locate best trained model
candidate_models = [
    'data/models/dentex_tufts_grounding_tool_cv_best/weights/best.pt',
    'data/models/dentex_grounding_tool_cv_best/weights/best.pt',
    'data/models/grounding_tool_cv_best/weights/best.pt',
    'data/models/dentex_tufts_cv_fold_0/weights/best.pt',
]
model_path = next((m for m in candidate_models if os.path.exists(m)), None)

if model_path:
    print(f'Loading model: {model_path}')
    model = YOLO(model_path)
    
    # Find sample test images
    test_images = glob.glob('data/yolo_*_cv/test/images/*.png') + glob.glob('data/yolo_*_cv/fold_0/images/val/*.png')
    if test_images:
        sample_img = test_images[0]
        print(f'Running inference on: {sample_img}')
        results = model.predict(sample_img, conf=0.25, imgsz=640)
        res_plotted = results[0].plot()
        
        plt.figure(figsize=(14, 7))
        plt.imshow(res_plotted)
        plt.title(f'YOLO Tooth Grounding Predictions — {Path(sample_img).name}', fontsize=14)
        plt.axis('off')
        plt.show()
    else:
        print('No sample images found in test/val directories.')
else:
    print('No trained model checkpoint found yet. Run training in Cell 6 first.')

## 9. Hugging Face Hub Artifact Sync
Uploads the complete `data/models/` tree (all fold checkpoints, curves, and evaluation tables) to `Reza-Nadimi/vlm-dental-models/yolo_cv`.

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
hf_repo = os.environ.get('HF_ARTIFACT_REPO', 'Reza-Nadimi/vlm-dental-models')

if hf_token and not hf_token.startswith('YOUR_'):
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=hf_repo, repo_type='model', exist_ok=True)
    print(f'Uploading all YOLO artifacts to {hf_repo}/yolo_cv...')
    api.upload_folder(
        folder_path='data/models',
        path_in_repo='yolo_cv',
        repo_id=hf_repo,
        repo_type='model',
        commit_message='Sync YOLO 5-Fold Cross-Validation models and metrics',
    )
    print('✅ Upload complete!')
else:
    print('⚠️ HF_TOKEN not set or invalid. Skipping automatic upload.')